Titre projet info

Import des données

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from fonctions import ajouter_colonne_statut_precis

# Import des données individuelles
url_individu = "https://object.data.gouv.fr/ministere-culture/FREQ_MUSEES/ENTREES_ET_CATEGORIES_DE_PUBLIC.csv"

df_individu = pd.read_csv(url_individu, sep=';')
df_individu = df_individu.loc[:,['IDPatrimostat', 'IDMuseofile']]
df_individu = df_individu.rename(columns={'IDPatrimostat' : 'REF DU MUSEE'})
df_individu

Import de la seconde base de données

In [ ]:
# Import des données totales
url = "https://static.data.gouv.fr/resources/frequentation-des-musees-de-france-1/20250827-121955/frequentation-totale-mdf-2001-a-2016-data-def9.xlsx"
df_totale = pd.read_excel(url, sheet_name=None)

# Pour voir les noms des feuilles disponibles
print(df_totale.keys())

# Pour accéder à une table précise
df_freq_totale = df_totale['FREQUENTATION TOTALE']


# On veut ajouter les idmuseofiles pour la suite de l'étude
df_freq_totale = df_freq_totale.merge(df_individu, on="REF DU MUSEE", how="left")  
print(df_freq_totale.head())

df_freq_gratuite = df_totale['FREQ GRATUITE']
#print(df_freq_gratuite.head())

df_freq_payante = df_totale['FREQ PAYANTE']
#print(df_freq_payante.head())

In [ ]:
nb_musees = len(df_freq_totale)
nb_musees

Dans la deuxième base, le tableau "payant" n'a pas les mêmes noms de variables que les deux autres.

In [ ]:
df_freq_payante = df_freq_payante.rename(columns={"REF MUSEE": "REF DU MUSEE",})
df_freq_totale = df_freq_totale.rename(columns={"NEW REGIONS": "NOMREG",})
df_freq_gratuite = df_freq_gratuite.rename(columns={"NEW REGIONS": "NOMREG",})

Nous souhaitons transformer la deuxième base de données pour avoir une seule variable année.

In [ ]:

# Colonnes identifiantes à conserver
id_vars = ["REF DU MUSEE", "NOMREG", "NOM DU MUSEE", "VILLE", "Fréquentation"]
id_vars_1 = ["REF DU MUSEE", "NOMREG", "NOM DU MUSEE", "VILLE", "Fréquentation", "IDMuseofile"]

# Colonnes années
annee_vars = [str(y) for y in range(2001, 2017)]

# Passage en format long
df_freq_totale = df_freq_totale.melt(
    id_vars = id_vars_1,
    value_vars = annee_vars,
    var_name = "annee",
    value_name = "frequentation"
)

df_freq_gratuite = df_freq_gratuite.melt(
    id_vars = id_vars,
    value_vars = annee_vars,
    var_name = "annee",
    value_name = "frequentation"
)

df_freq_payante = df_freq_payante.melt(
    id_vars = id_vars,
    value_vars = annee_vars,
    var_name = "annee",
    value_name = "frequentation"
)

print(df_freq_totale.head())
print(df_freq_gratuite.head())
print(df_freq_payante.head())

Nous souhaitons connaître le nombre de musée n'ayant jamais communiqué leur fréquentation entre 2001 et 2016.

In [ ]:
# Quelles valeurs, autres que numériques, peut prendre la fréquentation?
valeurs_non_numeriques = (
    df_freq_totale.loc[
        pd.to_numeric(df_freq_totale["frequentation"], errors="coerce").isna(),
        "frequentation"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(sorted(valeurs_non_numeriques))

Combien de musées présentent ces modalités au moins un fois ?

In [ ]:
freq = df_freq_totale["frequentation"]

# On garde les lignes où la valeur est NA ou n'est pas un nombre
ligne = freq.isna() | pd.to_numeric(freq, errors="coerce").isna()

# On crée une colonne avec les modalités
df_temp = df_freq_totale.loc[ligne, ["REF DU MUSEE", "frequentation"]].copy()

# On remplace les NA par le texte "NA"
df_temp["frequentation"] = df_temp["frequentation"].fillna("NA")

# On compte le nombre de musées différents pour chaque modalité
resultat = df_temp.groupby("frequentation")["REF DU MUSEE"].nunique()

print(resultat)

Nous souhaitons maintenant connaître le nombre de musées présentant,toutes les années, une de ces modalités non numériques.

In [ ]:
# On marque les lignes où la fréquentation est NA ou non numérique
df_freq_totale["speciale"] = freq.isna() | pd.to_numeric(freq, errors="coerce").isna()

# Pour chaque musée, on regarde si toutes les lignes sont spéciales
resultat_toutes_annees = df_freq_totale.groupby("REF DU MUSEE")["speciale"].all()

# Nombre de musées concernés
nb_musees_toutes_annees = resultat_toutes_annees.sum()

pourcentage = (nb_musees_toutes_annees / nb_musees) * 100

print(f"{nb_musees_toutes_annees} musées n'ont aucune fréquentation renseignée entre 2001 et 2016, soit {pourcentage:.2f}%.")

On peut supprimer les lignes vides (non réponse totale)

In [ ]:
# Suppression des lignes où toutes les colonnes sont vides
df_freq_totale = df_freq_totale.dropna(how='all')
df_freq_gratuite = df_freq_gratuite.dropna(how='all')
df_freq_payante = df_freq_payante.dropna(how='all')

On souhaite, pour la suite des analyses, créer deux colonnes : l'une avec les statuts du musée et l'autre avec seulement des variables numériques (on remplace par NaN les valeurs non numériques)

In [ ]:
# Création de la colonne numérique seulement
df_freq_totale["freq_net"] = pd.to_numeric(df_freq_totale["frequentation"], errors="coerce")
df_freq_payante["freq_net"] = pd.to_numeric(df_freq_payante["frequentation"], errors="coerce")
df_freq_gratuite["freq_net"] = pd.to_numeric(df_freq_gratuite["frequentation"], errors="coerce")

# Correction d'une coquille (Retrait d'appelaltion))
df_freq_totale["frequentation"] = df_freq_totale["frequentation"].replace("Retrait d'appelaltion", "Retrait d'appellation")
df_freq_payante["frequentation"] = df_freq_payante["frequentation"].replace("Retrait d'appelaltion", "Retrait d'appellation")
df_freq_gratuite["frequentation"] = df_freq_gratuite["frequentation"].replace("Retrait d'appelaltion", "Retrait d'appellation")

# Création de la nouvelle colonne
df_freq_totale = ajouter_colonne_statut_precis(df_freq_totale)
df_freq_payante = ajouter_colonne_statut_precis(df_freq_payante)
df_freq_gratuite = ajouter_colonne_statut_precis(df_freq_gratuite)

# Distribution des statuts
print(df_freq_totale["Statut"].value_counts(dropna=False)) # pas de Nan dans tous les cas

Import de la seconde base de données sur les types de musées

In [ ]:
df_freq_totale.head()

In [ ]:
url_types_musees = "https://object.data.gouv.fr/ministere-culture/POP/museofile.csv"

# On utilise le symbole pipe | (Alt Gr + 6 sur un clavier français)
df_types_musees = pd.read_csv(url_types_musees, sep='|')

df_types_musees = df_types_musees[['Identifiant','Domaine_thematique']]
df_types_musees = df_types_musees.rename(columns={'Identifiant' : 'IDMuseofile'})

Merge des bases de données pour ajouter les informations sur les thématiques du musées

In [ ]:
df_freq_totale = df_freq_totale.merge(df_types_musees, on="IDMuseofile", how="left")  
print(df_freq_totale.head())

La colonne Domaine_thematique peut contenir plusieurs thématiques : "Archéologie;Arts décoratifs;Art moderne" par exemple. Nous aimerions créer des colonnes binaire pour chaque thématique (1 si le musées aborde cette thématique 0 sinon).

In [ ]:
# 1. On sépare les thématiques et on crée les colonnes binaires
# sep=';' indique que les thématiques sont séparées par des points-virgules
colonnes_binaires = df_freq_totale['Domaine_thematique'].str.get_dummies(sep=';')

# 2. On joint ces nouvelles colonnes à votre DataFrame d'origine
df_final = pd.concat([df_freq_totale, colonnes_binaires], axis=1)

print(df_final.head())
